# 哨兵机器人自主决策行为树：自定义节点使用说明

> 本手册描述 `rmuc_2026.xml` 中出现的所有 **项目自定义节点**（非 BT.CPP 内置控制节点），给出端口、语义、返回值约定与实现要点。  
> ⚠️ 消息已从单一 `RMUC.msg` 拆分为 **8 个独立小消息**，各自拥有独立话题。详见 `RMUC_Msg_Split_Doc.ipynb`。

## 1. 规则约束要点

> 用于理解行为树的策略设计背景。

| 主题 | 规则要点 |
|------|---------|
| **比赛阶段** | 七分钟比赛阶段（`stage_remain_time` 以秒计）为主循环执行条件；非比赛阶段回家并禁用发射 |
| **姿态系统** | 进攻/防御/移动三种姿态，切换冷却 **5 秒**；单局累计在某姿态超过 **3 分钟**，该姿态效果下降 |
| **脱战** | 存活状态下连续 **6 秒** 未发射弹丸且未被扣血；远程补血/远程补弹仅在脱战状态可用 |
| **补给区回血** | 占领己方补给区增益点获得每秒上限血量 **10%** 回血；开始 4 分钟后，若脱战且占领补给区，提升至每秒 **25%**（非脱战立即失效） |
| **允许发弹量** | 初始 **300**；可在补给区/基地增益点/前哨站增益点兑换；远程兑换成功后 **6 秒**生效；补给区每分钟占领一次累计 **+100 发**（可累积） |
| **复活与虚弱** | 正常读条复活后进入"虚弱"：发射机构锁定、无法占领增益点；接触可占领的前哨站/基地/补给区模块卡即可解除虚弱 |
| **己方堡垒** | 提供防御增益与射击热量冷却增益，以及"储备允许发弹量"；强度与 Δ（己方基地血量上限 − 现有基地血量）相关 |
| **敌方堡垒** | 比赛进行 3 分钟且对方前哨站被击毁后可占领；占领期间可能获得高易伤，属于 **高风险目标** |

## 2. 黑板（Blackboard）关键字段约定

> `PerceptionAndBlackboard` 子树通过 **5 个独立话题** 订阅数据，再由 `ParseSentryBlackboard` 解析写入黑板。

| 字段 | 含义 | 来源话题 |
|:---|:---|:---|
| `{game_status}` | 原始比赛状态消息 | `/game_status` |
| `{robot_status}` | 原始机器人状态消息 | `/robot_status` |
| `{rfid.status}` | 原始 RFID 状态消息 | `/rfid_status` |
| `{radar.tracks}` | 原始雷达跟踪消息 | `/radar/enemy_tracks` |
| `{pose.x}` / `{pose.y}` / `{pose.yaw}` | 自身位姿 | `/robot_position` |
| `{is_at_nav_goal}` | 是否到达导航目标 (bool) | `/robot_position` |
| `{time.now_ms}` | 当前时间戳 (ms) | `/game_status` |
| `{game.remain_s}` / `{game.elapsed_s}` | 比赛剩余/已进行时长（秒） | ParseSentryBlackboard |
| `{hp.cur}` / `{hp.max}` | 当前 / 上限血量 | ParseSentryBlackboard |
| `{heat.cur}` | 当前射击热量 | ParseSentryBlackboard |
| `{ammo.allow}` / `{ammo.left}` | 剩余允许发弹量 / 物理弹丸 | ParseSentryBlackboard |
| `{economy.coins}` | 队伍金币 | ParseSentryBlackboard |
| `{state.disengaged}` / `{state.disengage_cd_s}` | 是否脱战 / 脱战倒计时 | ParseSentryBlackboard |
| `{state.is_dead}` / `{state.is_weak}` | 是否战亡 / 虚弱 | ParseSentryBlackboard |
| `{base.hp.cur}` / `{base.hp.max}` | 己方基地血量 | ParseSentryBlackboard |
| `{outpost.alive}` | 己方前哨站是否存活 | ParseSentryBlackboard |
| `{combat.has_target}` / `{combat.best_target}` | 目标检测 | ParseSentryBlackboard |
| `{threat.base}` | 基地威胁评估 | ParseSentryBlackboard |

## 3. 自定义节点目录

> BT.CPP 内置节点（Sequence / Fallback / ReactiveSequence / ReactiveFallback / WhileDoElse / RateController 等）不在此列。

---

### 3.1 订阅节点：SubGameStatus / SubRobotStatus / SubRFIDStatus / SubRobotPosition / SubRadarTracks

- **类型**：Action（RosTopicSubNode，订阅并写黑板）
- **每个节点订阅独立话题**：

| 节点 | 话题 | 消息类型 | 输出端口 |
|:---|:---|:---|:---|
| `RmucSubGameStatus` | `/game_status` | `RMUCGameStatus` | `game_status`, `now_ms` |
| `RmucSubRobotStatus` | `/robot_status` | `RMUCRobotStatus` | `robot_status` |
| `RmucSubRFIDStatus` | `/rfid_status` | `RMUCRFIDStatus` | `rfid_status` |
| `RmucSubRobotPosition` | `/robot_position` | `RMUCRobotPosition` | `pose_x/y/yaw`, `is_at_nav_goal` |
| `SubRadarTracks` | `/radar/enemy_tracks` | `RMUCEnemyTracks` | `radar_tracks` |

- **返回值**：正常情况下返回 `SUCCESS`
- **实现要点**：
  - **非阻塞**：每 tick 尽快返回 SUCCESS，内部缓存最新消息
  - 若消息长时间未更新，可设置降级标志供 `ParseSentryBlackboard` 使用

---

### 3.2 ParseSentryBlackboard

- **类型**：Action（SyncActionNode，解析裁判系统与感知，派生高层语义）
- **输入端口**：`game_status`(`RMUCGameStatus`) / `robot_status`(`RMUCRobotStatus`) / `radar_tracks`(`RMUCEnemyTracks`) / `pose_x` / `pose_y` / `now_ms`
- **输出端口**：`{game.remain_s}` / `{game.elapsed_s}`、`{hp.*}`、`{heat.cur}`、`{ammo.*}`、`{economy.*}`、`{state.*}`、`{combat.*}`、`{threat.*}` 等
- **返回值**：成功解析返回 `SUCCESS`
- **实现要点**：
  - 建议在此节点统一做：**脱战判定**、**虚弱判定**、**基地威胁评估**、**目标筛选预处理**
  - `stage_remain_time` 直接从 `RMUCGameStatus` 读取；`elapsed` = `420 - remain`
  - `rfid_status` 不再作为本节点输入（RFID 由条件节点直接从黑板读取 `RMUCRFIDStatus`）

---

### 3.3 InitSentryConfig

- **类型**：Action（SyncActionNode，输出配置到黑板）
- **输出端口**：
  - 各关键点坐标：`home` / `supply` / `base_buff` / `outpost_buff` / `fortress` / 高地 / 巡逻点等
  - 阈值：`hp_critical` / `hp_low` / `hp_safe`、`heat_high` / `heat_critical`、`ammo_low` / `ammo_target` 等
- **返回值**：返回 `SUCCESS`
- **实现要点**：
  - 默认值均为 0.0 或保守阈值；**务必根据实际地图坐标系与机器人能力标定**
  - `arrive_radius` 建议与导航定位精度和模块卡死区大小匹配

### 3.4 DecidePosture

- **类型**：Action（决定哨兵姿态）
- **输入端口**：`hp_cur` / `hp_max`、`heat_cur`、`has_target`、`base_threat`、`is_disengaged`、`stage_elapsed_time`、`now_ms`
- **输出端口**：`posture_out` (int)

| 值 | 姿态 |
|----|------|
| 1 | 进攻姿态 |
| 2 | 防御姿态 |
| 3 | 移动姿态 |

- **返回值**：返回 `SUCCESS`（仅输出姿态）
- **实现要点**：
  - 内置 **5 秒** 切换冷却，避免频繁抖动
  - 建议策略：有稳定交战窗口→进攻；守家/血量低→防御；长距离转场/无目标→移动
  - 可在内部记录各姿态累计时间，超过 3 分钟后按规则降低该姿态收益

---

### 3.5 DecideEconomyCmd

- **类型**：Action（决定远程补血/补弹与允许发弹量目标）
- **输入端口**：`hp_cur` / `hp_max`、`ammo_allow` / `ammo_target` / `ammo_low`、`is_disengaged`、`can_remote_*`、`team_coins`、`stage_remain_time`、`base_threat`、`allow_ammo_target_in`
- **输出端口**：`allow_ammo_target_out`（单调不减）、`trigger_remote_ammo` (0/1)、`trigger_remote_hp` (0/1)、`enable_big_energy` (0/1)
- **返回值**：返回 `SUCCESS`
- **实现要点**：
  - 远程补血/补弹必须满足：**脱战** + `can_remote_*` + **金币足够** + 预计 6 秒内生存概率高（建议 `base_threat==false` 且无近敌）
  - `allow_ammo_target_out` 按协议要求必须 **单调递增**
  - `stage_remain_time` 可调整经济决策：越接近终局，远程补血/立即复活性价比可能更高

---

### 3.6 DecideRespawnCmd

- **类型**：Action（决定确认复活 / 兑换立即复活）
- **输入端口**：`is_dead`、`robot_status`（含复活读条状态/是否可复活）、`team_coins`、`stage_remain_time`、`base_hp_cur` / `base_hp_max`、`base_threat`
- **输出端口**：`confirm_respawn` (0/1)、`confirm_instant_respawn` (0/1)
- **返回值**：返回 `SUCCESS`
- **实现要点**：
  - 优先级：若终局且守家压力大且金币允许→倾向 **立即复活**；否则走正常读条复活并尽快触卡解除虚弱
  - 注意协议处理顺序：服务器按从低位到高位依次处理 bit 指令，遇到金币不足会忽视后续指令

---

### 3.7 SentryCmdMux（0x0120）

- **类型**：Action（打包并发送"哨兵自主决策指令"到裁判系统）
- **输入端口**：`posture`、`confirm_respawn`、`confirm_instant_respawn`、`allow_ammo_target`、`trigger_remote_ammo`、`trigger_remote_hp`、`enable_big_energy`
- **inout 端口**：`cmd_state`（内部状态，维护单调计数与边沿触发）
- **返回值**：发送成功返回 `SUCCESS`；发送失败可返回 `FAILURE`

**协议 bit 布局**：

| Bit 范围 | 功能 |
|----------|------|
| bit 0 | 确认复活 |
| bit 1 | 确认兑换立即复活 |
| bit 2–12 | 允许发弹量目标值（单调递增） |
| bit 13–16 | 远程兑换发弹量请求次数（单调递增，每次 +1） |
| bit 17–20 | 远程兑换血量请求次数（单调递增，每次 +1） |
| bit 21–22 | 姿态（1 进攻 / 2 防御 / 3 移动） |
| bit 23 | 大能量机关确认（1 确认） |

- **实现要点**：
  - `trigger_remote_ammo` / `trigger_remote_hp` 建议做 **上升沿触发**：由 0→1 时把计数 +1 并发送，持续为 1 时不重复累加
  - `allow_ammo_target` 建议直接写"累计目标"，由节点内部保证单调与裁判系统一致性

### 3.8 导航目标选择：SelectObjective / SelectNearestResupplyStation / SelectNearestDispelCard / SelectSafeRetreatGoal

- **类型**：Action（输出导航目标点）
- **输入端口**：当前位置 `pose_x` / `pose_y` + 多个候选点坐标 +（可选）战局信息
- **输出端口**：`goal_x` / `goal_y` +（`SelectObjective` 额外输出 `objective_name`）
- **返回值**：找到目标返回 `SUCCESS`；找不到返回 `FAILURE`（让上层落入 PatrolAndScan）
- **实现要点**：
  - `SelectObjective` 建议固定优先级：**守家/堡垒 > 中央高地 > 梯形高地 > 其它地形跨越/巡逻**
  - 基地血量缺口 Δ 较大时，优先占领己方堡垒以获得更强的防御与冷却

---

### 3.9 条件检测：IsZoneCardDetected / IsAnyDispelCardDetected

- **类型**：Condition（场地交互模块 / 视觉标签检测）
- **输入端口**：`rfid_status` + `robot_status`（可选）
- **参数**：`zone`（字符串枚举：`SUPPLY` / `BASE_BUFF` / `OUTPOST_BUFF` / `FORTRESS` ...）
- **返回值**：满足条件返回 `SUCCESS`，否则 `FAILURE`
- **实现要点**：
  - 模块卡存在死区与延迟，占领失效有 **2 秒** 延迟；实现应具备 **去抖与短暂丢失容忍**
  - `IsAnyDispelCardDetected` 用于虚弱解除：检测到任意可占领的补给区/基地/前哨站卡即返回 true

---

### 3.10 区域保持：HoldAndHeal / HoldForSupplyAmmoTick / HoldObjective

- **类型**：Action（在目标区保持并等待规则收益）

| 节点 | 功能 | 完成条件 |
|------|------|---------|
| `HoldAndHeal` | 在补给区等待回血（4 分钟后脱战可获 25%/s 高速回血） | 血量恢复至 `hp_safe` |
| `HoldForSupplyAmmoTick` | 在补给区等待"每分钟 +100 允许发弹量"的免费获取 | 达到 `ammo_target` 或累积完成 |
| `HoldObjective` | 占点保持一段时间 | 计时结束，或 `base_threat` / `has_target` 触发提前结束 |

- **返回值**：等待中返回 `RUNNING`；完成返回 `SUCCESS`
- **实现要点**：
  - 等待过程中可允许云台扫描（`stop_gimbal_scan=false`），但应避免误触发射导致脱战失效

---

### 3.11 交战链路：SelectBestTarget / AimAtTarget / FireBurst / IsFireWindowOk

| 节点 | 类型 | 功能 |
|------|------|------|
| `SelectBestTarget` | Action | 从 `radar_tracks` 选择目标（可偏向威胁基地、距离近、血量低等） |
| `AimAtTarget` | Action | 云台对准目标 |
| `FireBurst` | Action | 按 `burst_ms` / `pause_ms` 节奏开火以控热 |
| `IsFireWindowOk` | Condition | 热量、允许发弹量、虚弱状态等门控 |

- **返回值**：`SelectBestTarget` → `SUCCESS`；`AimAtTarget` / `FireBurst` → `RUNNING`；`IsFireWindowOk` → `SUCCESS` / `FAILURE`
- **实现要点**：
  - `FireBurst` 建议结合热量模型与上限，避免超限惩罚
  - 若系统支持动态量（上一发射速/热量冷却速率），可进一步自适应 `burst_ms` / `pause_ms`

## 附录

### A. 行为树集成建议

1. **CommandHub 子树** 建议始终运行，保证姿态与远程兑换请求在任何战术分支中都能持续更新
2. 若工程已有"导航到点 + 到达判定 + 微动触卡"封装，可直接替换 `HealPlan` / `AmmoPlan` 的相关节点
3. **map 坐标与关键点** 必须在上场前标定；建议把 `InitSentryConfig` 的默认值替换为 YAML / 参数服务器加载
4. 模块卡可能存在延迟与死区，占点相关 Condition 节点 **务必做滤波去抖**

### B. 消息类型与话题对照

| 消息类型 | 话题 | 方向 | 使用的 RosNodeParams |
|:---|:---|:---:|:---|
| `RMUCGameStatus` | `/game_status` | 📥 | `params_game_status` |
| `RMUCRobotStatus` | `/robot_status` | 📥 | `params_robot_status` |
| `RMUCRFIDStatus` | `/rfid_status` | 📥 | `params_rfid_status` |
| `RMUCRobotPosition` | `/robot_position` | 📥 | `params_robot_position` |
| `RMUCEnemyTracks` | `/radar/enemy_tracks` | 📥 | `params_radar` |
| `RMUCSentryCmd` | `/sentry_cmd` | 📤 | `params_sentry_cmd` |
| `RMUCRobotControl` | `/robot_control` | 📤 | `params_robot_ctrl` |
| `RMUCNavControlCmd` | `/nav_control_cmd` | 📤 | `params_nav_cmd` |

> 详细的拆分说明、踩坑记录见 `RMUC_Msg_Split_Doc.ipynb`。